# Labelled examples

In [1]:
import pandas as pd
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *

In [ ]:
#helper functions
def sort_appeal_type(x):
    pref_order = ["DREF Operation Final Report", "DREF Operation Update", "DREF Operation", "Operations Update"]
    for appealtype in pref_order:
        if appealtype in list(x['appealType']):
            return x[x['appealType'] == appealtype]

def select_report_by_appealCode(appeal_code, report_df):
    report = report_df.where(report_df.appealCode == appeal_code).dropna()
    return report

def print_report(appeal_code, report, text_field="nathaz_text"):
    print(f"{appeal_code}: {report.reportDate}")
    text = report.iloc[0][text_field]
    # Convert to list if it's a string that looks like a list
    if pd.notna(text):
        if isinstance(text, str) and text.strip().startswith("[") and text.strip().endswith("]"):
            try:
                text_list = ast.literal_eval(text)
            except Exception:
                text_list = [text]  # fallback if parsing fails
        else:
            text_list = [text]
    else:
        text_list = []
    
    # Print each sentence on a new line
    for sentence in text_list:
        print(sentence)
    # print("\n".join(text))

def add_report_date(report, impact_dict_or_list):
    if isinstance(impact_dict_or_list, dict):
        impact_dict_or_list["reportDate"] = report.date
    elif isinstance(impact_dict_or_list, list):
        for i in range(len(impact_dict_or_list)):
            impact_dict_or_list[i]["reportDate"] = report.date
    else:
        raise TypeError("impact_dict_or_list must be a dictionary or a list of dictionaries")
    return impact_dict_or_list

def download_report(report, savelocation):
    link = report["reportLink"]
    savename = report["origType"]+".pdf"
    r = requests.get(link)
    with open(savelocation + savename, 'wb') as f:
        f.write(r.content)

def report_dict_to_df(labelled_reports_dict):
    df_list = []
    for k,v in labelled_reports_dict.items():
        df = pd.DataFrame(v)
        df['appealCode'] = k
        df_list.append(df)
    df_all = pd.concat(df_list)
    df_all.reset_index(inplace=True, drop=True)
    return df_all

In [17]:
#Load data

# file_path = DATA_IN_JSONS + 'filtered_report_types_nat_hazards_nathaz_text.json'#'all_ifrc_reports_info_processed_extended.json' #'all_ifrc_reports_info_processed_extended_format_nb_std_units.json'

# # Open and read the JSON file
# with open(file_path, 'r') as json_file:
#     filtered_reports = json.load(json_file)
# filtered_reports = pd.DataFrame(filtered_reports)
# filtered_reports['date'] = pd.to_datetime(filtered_reports["date"], format="%d/%m/%Y")

file_path = DATA_IN_JSONS + 'preproc_filtered_report_types_nat_hazards_bugfix_v180925.csv'
filtered_reports = pd.read_csv(file_path)
filtered_reports["reportDate"] = pd.to_datetime(filtered_reports["reportDate"])
# filtered_reports['date'] = pd.to_datetime(filtered_reports["date"], format="%d/%m/%Y")

In [18]:
filtered_reports.head(3)

,reportName,disasterType,dateTime,reportLink,location,appealCode,appealType,origType,pdfDownloaded,text,...,disasterTypeReclassified,naturalHazard,text_processed,sentences,language,iso_code,reportDate,nathaz_text,hazards_found_kw,secondaryDisasterType
0,Cuba - Cyclone Hurricane Rafael (MDRCU011),Cyclone,2025-07-08T10:01:00+02:00,https://go-api.ifrc.org/api/downloadfile/91656...,Cuba,MDRCU011,DREF Operation Update,MDRCU011ou2,1,DREF Operational Update\nCuba: Hurricane Rafae...,...,"Cyclone, Hurricane",1,DREF Operational Update Cuba: Hurricane Rafael...,['DREF Operational Update Cuba: Hurricane Rafa...,en,CUB,2025-07-08,['DREF Operational Update Cuba: Hurricane Rafa...,"['Earthquake', 'Mass movement', 'Flood', 'Trop...",NaN
1,DRC - Floods (MDRCD046),Flood,2025-07-03T18:01:00+02:00,https://go-api.ifrc.org/api/downloadfile/91611...,"Congo, The Democratic Republic Of The",MDRCD046,DREF Operation Update,MDRCD046du1,1,DREF Operational Update\nDemocratic Republic o...,...,Flood,1,DREF Operational Update Democratic Republic of...,['DREF Operational Update Democratic Republic ...,en,COD,2025-07-03,['DREF Operational Update Democratic Republic ...,"['Flood', 'Epidemics', 'Conflict']",NaN
2,Ethiopia - Landslides and Floods (MDRET036),Landslide,2025-07-03T11:16:00+02:00,https://go-api.ifrc.org/api/downloadfile/91580...,Ethiopia,MDRET036,Operations Update,MDRET036eu2,1,\n \n \n \n1 \n \nOPERATION UPDATE \nCountry|...,...,"Flood, Mass movement",1,1 OPERATION UPDATE Country| Emergency Emergenc...,['1 OPERATION UPDATE Country| Emergency Emerge...,en,ETH,2025-07-03,"['SITUATION ANALYSIS 1.', 'Description of the ...","['Wildfire', 'Mass movement', 'Flood', 'Epidem...",NaN


In [19]:
#Previously labelled reports 
labelled_v1_reports = pd.read_csv(DATA_LABELLED + 'labelled_8reports_impact_laura.csv', encoding='utf-8')
labelled_v1_reports["reportDate"] = pd.to_datetime(labelled_v1_reports["reportDate"])

In [51]:
labelled_v1_reports.loc[labelled_v1_reports["appealCode"]=="MDRBD022"]

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode
0,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,['DREF operation n MDRBD022 Glide n FL-2019-00...,['Bangladesh'],NaN,2019,7.0,18.0,2019.0,11.0,18.0,['Flood'],MDRBD022
1,2019-07-19,Affected People,1000000.0,people,approx,NaN,NaN,['While the monsoon season normally brings ann...,"['Nepal', 'India']",NaN,2019,NaN,NaN,NaN,NaN,NaN,['Flood'],MDRBD022
2,2019-07-19,Affected People,NaN,people,approx,2100000.0,NaN,['According to National disaster response coor...,['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
3,2019-07-19,Residential Buildings,10000.0,houses destroyed,approx,NaN,NaN,['According to National disaster response coor...,['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
4,2019-07-19,Crop Production and Forestry,14733.0,hectares of crops,approx,NaN,NaN,['According to National disaster response coor...,['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
5,2019-07-19,Other Infrastructural impact,NaN,NaN,NaN,NaN,NaN,['It is also reported that embankments have be...,['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
6,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,"['of affected population 2,176,519 No.', 'Six ...",['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
7,2019-07-19,Residential Buildings,3988.0,houses damaged,exact,NaN,NaN,"['of fully damaged house 3,988 No.', 'Six days...",['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
8,2019-07-19,Residential Buildings,98571.0,houses partially damaged,exact,NaN,NaN,"['of partially damaged house 98,571 No.', 'Six...",['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022
9,2019-07-19,Displaced People,31818.0,people,exact,NaN,NaN,"['of people who have moved to safe shelter 31,...",['Bangladesh'],"['Kurigram district', 'Gaibandha district', 'L...",2019,7.0,16.0,NaN,NaN,NaN,['Flood'],MDRBD022


In [57]:
appealCode_list = ["MDRCN006", "MDRBD022", "MDRYE011", "MDRS2001", "MDRIQ014", "MDRGN015", "MDRSV012", "MDRMY003", "MDRKE058"]

# Select the reports corresponding to the ones already labelled 
keys = labelled_v1_reports[['appealCode', 'reportDate']].drop_duplicates()
reports_to_label_1 = filtered_reports.merge(keys, on=['appealCode', 'reportDate'], how='inner')
appealCode_labelled = reports_to_label_1.appealCode.unique()

# For the rest used the previous selection method 
appealCode_select = list(set(appealCode_list) - set(appealCode_labelled))
reports_to_label_all = filtered_reports.loc[filtered_reports.appealCode.isin(appealCode_select)]
reports_to_label_all = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: sort_appeal_type(x)).reset_index(drop=True)
reports_to_label_all.reportDate = pd.to_datetime(reports_to_label_all.reportDate, dayfirst=True)
reports_to_label_2 = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: x.sort_values('reportDate', ascending=False).head(1)).reset_index(drop=True)

reports_to_label = pd.concat([reports_to_label_1, reports_to_label_2], ignore_index=True)

#check that everything is there
print(f"Number appealCodes: {len(appealCode_list)}, number reports: {len(reports_to_label)}")
missing_appealCode = list(set(appealCode_list) - set(reports_to_label.appealCode))
print(f"Missing appealCodes: {missing_appealCode}")

Number appealCodes: 9, number reports: 9
Missing appealCodes: []


C:\Users\lhasbini\AppData\Local\Temp\ipykernel_9380\2094896911.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reports_to_label_all = reports_to_label_all.groupby('appealCode', as_index=False).apply(lambda x: sort_appeal_type(x)).reset_index(drop=True)
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_9380\2094896911.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reports_to_label_2 = reports_to_label_all.grou

In [45]:
labelled_impact_reports_dict = {} # dict to store labelled reports
#empty dict structure to store results
# labelled_impact_reports_dict["appealCode"]=[
#     {"reportDate": None,
#      "impactSubtype" : None,
#      "impactValue" : None, 
#      "impactUnit" : None, 
#      "impactValuePrecision" : None, 
#      "impactValueMin" : None, 
#      "impactValueMax" : None, 
#      "annotation" : None,
#      "country" : None,
#      "location" : None,
#      "startYear" : None,
#      "startMonth" : None,
#      "startDay" : None,
#      "endYear" : None,
#      "endMonth" : None,
#      "endDay" : None,
#      "hazards" : None,
#     },
# ]

# MDRCN006

In [88]:
iappeal = "MDRCN006"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==appeal]
print_report(iappeal, ireport)

MDRCN006: 6   2019-03-14
Name: reportDate, dtype: datetime64[ns]
DREF operation Operation n° MDRCN006 Date of Issue: 15 February 2019 Glide number: TC-2018-000110-CHN Date of disaster: 7 July 2018 Operation start date: 15 July 2018 Operation end date: 15 November 2018 Host National Society: Red Cross Society of China (RCSC) Operation budget: CHF 381,563 Number of people affected: 1,381,000 Number of people assisted: 27,800 persons0F1 N° of National Societies involved in the operation: International Federation of Red Cross and Red Crescent Societies (IFRC) N° of other partner organizations involved in the operation: National Disaster Reduction Commission, Ministry of Emergency Management of People Republic of China A.
SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.
In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for fou

In [89]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1381000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 222000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Injured People",
     "impactValue" : 22000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 900, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 29000, 
     "impactValueMax" : None, 
     "annotation" : ['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.', 
                     'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died 222,000 had taken emergency resettlement 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan that includes 15 cities and 70 counties more than 900 houses collapsed, and 29,000 houses damaged.'],
     "country" : ["China"],
     "location" : ["Deyang", "Mianyang", "Guangyuan"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 13,
     "hazards" : ["Flood", "Tropical storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 36900, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A total of 36,900 hectares of crops were also affected by the flood.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Other Economic and Livelihood Impacts",
     "impactValue" : None, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 792000000, 
     "impactValueMax" : None, 
     "annotation" : ['The direct economic loss was estimated to be over 5.3 billion Yuan approximately CHF 792 million.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Affected People",
     "impactValue" : 1519000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 12, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Missing People",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Displaced People",
     "impactValue" : 30000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The area of Tianshui, Zhangye, Pingliang including 10 cities and 46 counties were flooded, and affected 1,519,000 people where 12 died 4 missing and 30,000 were evacuated.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses collapsed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2300, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 19000, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'More than 2,300 houses collapsed, and 19,000 were damaged to varying degrees.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Other Economic and Livelihood Impacts",
     "impactValue" : 538000000, 
     "impactUnit" : "CHF", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.', 
                     'The direct economic loss was estimated 3.6 billion Yuan approximately CHF 538 million.'],
     "country" : ["China"],
     "location" : ["Tianshui", "Zhangye", "Pingliang"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazards" : ["Flood", "Convective storm"]
    }, 
    {"reportDate": "2019-03-14",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : "houses", 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to this rapid assessment, heavy rainfall had resulted in a large number of seriously damaged houses that have continued to collapse in these two provinces.'],
     "country" : ["China"],
     "location" : ["Sichuan", "southeast region of Gansu"],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [90]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRBD022

In [91]:
iappeal = "MDRBD022"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRBD022: 4   2019-07-19
Name: reportDate, dtype: datetime64[ns]
DREF operation n° MDRBD022 Glide n° FL-2019-000079-BGD Date of issue: 18 July 2019 Expected timeframe: 4 months Expected end date: 18 November 2019 Category allocated to the of the disaster or crisis: Orange DREF allocated: CHF 452,439 Total number of people affected: 2,176,519 Number of people to be assisted: 50,000 Host National Society presence (n° of volunteers, staff, branches): Bangladesh Red Crescent Society (BDRCS) – over 575 Red Crescent volunteers and 100 staff mobilized.
Red Cross Red Crescent Movement partners actively involved in the operation: American Red Cross, British Red Cross, Danish Red Cross, German Red Cross, Swedish Red Cross, Swiss Red Cross, Italian Red Cross, Turkish Red Crescent, Qatar Red Crescent and the International Committee of the Red Cross (ICRC).
Other partner organizations actively involved in the operation: Government of Bangladesh, UN RC, UNICEF, WFP, Terre das hommes (TdH), Oxfam, ST

In [92]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['DREF operation n MDRBD022 Glide n FL-2019-000079-BGD Date of issue 18 July 2019 Expected timeframe 4 months Expected end date 18 November 2019 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,439 Total number of people affected 2,176,519 Number of people to be assisted 50,000 Host National Society presence (n of volunteers, staff, branches) Bangladesh Red Crescent Society (BDRCS) over 575 Red Crescent volunteers and 100 staff mobilized.'],
     "country" : ["Bangladesh"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 18,
     "endYear" : 2019,
     "endMonth" : 11,
     "endDay" : 18,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1000000, 
     "impactValueMax" : None, 
     "annotation" : ['While the monsoon season normally brings annual floods to the country and wider region, this year, widespread flooding in upstream countries, Nepal and India, where millions of people have been severely impacted, have meant that the scale of the flooding this year has been significantly exacerbated.'],
     "country" : ["Nepal", "India"],
     "location" : None,
     "startYear" : 2019,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 2100000, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 100000, 
     "impactUnit" : "houses destroyed", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to National disaster response coordination centre (NDRCC) report dated 16 July more than 2.1 million people have been affected in 21 districts, around 100,000 houses destroyed and about 14,733 hectares of crop land damaged.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"] 
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['It is also reported that embankments have been damaged and inundated in new areas.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Affected People",
     "impactValue" : 2176519, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of affected population 2,176,519 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 3988, 
     "impactUnit" : "houses fully damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of fully damaged house 3,988 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 98571, 
     "impactUnit" : "houses partially damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of partially damaged house 98,571 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Displaced People",
     "impactValue" : 31818, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 14733, 
     "impactUnit" : "hectares of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of people who have moved to safe shelter 31,818 Amount of crop land damaged (Hectare) 14,733 No.', 
                     'Six days of heavy rain and onrush of upstream waters have flooded low-lying areas of Kurigram, Gaibandha, Lalmonirhat, Chattogram, Bandarban, Rangamati, Sylhet, Sunamganj, Bogura, Nilphamari, Khagrachari, Coxs Bazar, Feni, Netrokona, Sirajganj, Jamalpur, Tangail, Moulavibazar, Habiganj, Sherpur, and Brahmanbaria districts.'],
     "country" : ["Bangladesh"],
     "location" : ["Kurigram district", "Gaibandha district", "Lalmonirhat district", "Chattogram district", "Bandarban district", "Rangamati district", "Sylhet district", "Sunamganj district", "Bogura district", "Nilphamari district", "Khagrachari district", "Coxs Bazar district", "Feni district", "Netrokona district", "Sirajganj district", "Jamalpur district", "Tangail district", "Moulavibazar district", "Habiganj district", "Sherpur district", "Brahmanbaria district"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    },
    {"reportDate": "2019-07-19",
     "impactSubtype" : "Other Infrastructural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['of union affected 776 Emergency Plan of Action (EPoA) Bangladesh Floods P a g e 2 People watch as water from the swollen Teesta river gush into their neighbourhood in Gaddimari area of Lalmonirhats Hatibandha after a part of the protection embankment collapsed on 13 July afternoon.'],
     "country" : ["Bangladesh"],
     "location" : ["Gaddimari area",  "Lalmonirhats Hatibandha"],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : 13,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }
]

In [93]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)


# MDRYE011

In [94]:
iappeal = "MDRYE011"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRYE011: 2   2022-08-03
Name: reportDate, dtype: datetime64[ns]
P a g e | 1 Internal DREF Operation n° MDRYE011 Glide n°: FL-2022-000265-YEM Date of issue: 29/07/2022 Expected timeframe: 6 months Expected end date: 31/01/2023 Category allocated to the of the disaster or crisis: Orange DREF allocated: CHF 452,156 Total number of people affected: Approximately 76,790 people affected Number of people to be assisted: 19,509 people (2,787 HH) Governorates affected: Marib, Al Mahwit, Taiz, Ibb, Hadramawt, Al Bayda, Amran, Sadaa, Dhamar Al Hodeida Sana'a Hajjah, Al Mahra governorates Governorates targeted: Al Hodeida, Hajjah Hadramout, and Al- Mahra, Marib and Sana’a Governorates Operating National Society: Yemen Red Crescent Society has branches in all 22 Governates of Yemen, with 321 staff and 4,500 active volunteers, including 44 National Disaster Response trained team members, as well as trained first aid volunteers ready to deploy in case of emergency.
Red Cross Red Crescent Movement pa

In [95]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 76790, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["P a g e 1 Internal DREF Operation n MDRYE011 Glide n FL-2022-000265-YEM Date of issue 29/07/2022 Expected timeframe 6 months Expected end date 31/01/2023 Category allocated to the of the disaster or crisis Orange DREF allocated CHF 452,156 Total number of people affected Approximately 76,790 people affected Number of people to be assisted 19,509 people (2,787 HH) Governorates affected Marib, Al Mahwit, Taiz, Ibb, Hadramawt, Al Bayda, Amran, Sadaa, Dhamar Al Hodeida Sana'a Hajjah, Al Mahra governorates Governorates targeted Al Hodeida, Hajjah Hadramout, and Al- Mahra, Marib and Sanaa Governorates Operating National Society Yemen Red Crescent Society has branches in all 22 Governates of Yemen, with 321 staff and 4,500 active volunteers, including 44 National Disaster Response trained team members, as well as trained first aid volunteers ready to deploy in case of emergency."],
     "country" : ["Yemen"],
     "location" : ["Marib", "Al Mahwit", "Taiz", "Ibb", "Hadramawt", "Al Bayda", "Amran", "Sadaa", "Dhamar Al Hodeida Sana'a Hajjah", "Al Mahra", "Al Hodeida", "Hajjah Hadramout", "Marib", "Sanaa"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property."],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Informal settlements",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property."],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property."],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 3, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property.", 
                     '3 people died and 2 people were injured due to the heavy rain that led to the collapse of their house which consist of three floors and 3 families.'],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Injured People",
     "impactValue" : 2, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Situation analysis Description of the disaster On Saturday,23 July 2022, in Sana'a governorate, heavy rains led to floods causing extensive damage to public infrastructure, shelters for displaced people and other private property.", 
                     '3 people died and 2 people were injured due to the heavy rain that led to the collapse of their house which consist of three floors and 3 families.'],
     "country" : ["Yemen"],
     "location" : ["Sanaa governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 56, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Khamis camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 137, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Hasaba camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 116, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Aser camp"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 63, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['the heavy rains affected 56 families in Al-Khamis camp, 137 families at IDPs camp in Al-Hasaba, 116 families at Aser camp in addition to 63 families at Al-Tahreer Square.'],
     "country" : ["Yemen"],
     "location" : ["Al-Tahreer Square"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 299, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Saada Governorate,299 families were affected by heavy rains, Emergency Plan of Action (EPoA) Yemen Sanaa Floods P a g e 2 Internal and approximately 50 families of them were affected by heavy rains in the districts of Saada, Sahara, and Majaz.'],
     "country" : ["Yemen"],
     "location" : ["Saada Governorate"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 50, 
     "impactUnit" : "families", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Saada Governorate,299 families were affected by heavy rains, Emergency Plan of Action (EPoA) Yemen Sanaa Floods P a g e 2 Internal and approximately 50 families of them were affected by heavy rains in the districts of Saada, Sahara, and Majaz.'],
     "country" : ["Yemen"],
     "location" : ["Saada district", "Sahara district", "Majaz district"],
     "startYear" : 2022,
     "startMonth" : 7,
     "startDay" : 23,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 371, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 504, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Mahwit"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1085, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Hadramout"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1127, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Dhamar"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1211, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Bayda"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1239, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Saada"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 1610, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Ibb"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 2828, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al- Hodeida"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 3087, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Amran"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 3290, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Al-Dhalea"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 18039, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Marib"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 19600, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Taiz"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 20300, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Hajjah"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Affected People",
     "impactValue" : 2499, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.',],
     "country" : ["Yemen"],
     "location" : ["Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    },
    {"reportDate": "2022-08-03",
     "impactSubtype" : "Water Quality and Availability",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Based on YRCS and shelter cluster flood tracking and identification to date, the breakdown of people in need by governorate since June is shown in the below table Governorate Affected Households Affected people IDPs Site Al-Mahra 53 371 Al-Mahwit 72 504 Hadramout 155 1,085 2 Dhamar 161 1,127 Al-Bayda 173 1,211 Saada 177 1,239 Ibb 230 1,610 Al- Hodeida 404 2,828 13 Amran 441 3,087 Al-Dhalea 470 3,290 11 Marib 2,577 18,039 197 Taiz 2,800 19,600 Hajjah 2,900 20,300 22 Sanaa 357 2,499 2 Total 10,970 76,790 247 Of the total number of people affected, an estimated 17,000 across affected IDP sites have suffered total damage to tents and other belongings.', 
                     'Floods and storms have caused the total or partial destruction of tents, loss of personal belongings, food and essential household items, and damage to water tanks and sewage networks.'],
     "country" : ["Yemen"],
     "location" : ["Al-Mahra", "Al-Mahwit", "Hadramout", "Dhamar", "Al-Bayda", "Saada", "Ibb", "Al- Hodeida", "Amran", "Al-Dhalea", "Marib", "Taiz", "Hajjah", "Sana’a"],
     "startYear" : 2022,
     "startMonth" : 6,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Other storm"],
    }
]


In [96]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRS2001

In [97]:
iappeal = "MDRS2001"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRS2001: 0   2024-08-06
Name: reportDate, dtype: datetime64[ns]
SITUATION ANALYSIS Description of the crisis Hurricane Beryl emerged as a significant climate event, developing from a monitored tropical wave on June 25, 2024.
The storm rapidly intensified, becoming the first major hurricane of the 2024 Atlantic season and reaching unprecedented strength.
By June 29, 2024, Beryl had attained Category 4 status, setting a record as the earliest Category 4 hurricane in history.
The storm continued to strengthen, reaching Category 5 with maximum sustained winds of 270 km/h by July 1, 2024.
This highlights the increasing severity and unpredictability of hurricanes in the Caribbean, exacerbated by rising sea temperatures.
Beryl's impact was devastating across multiple Caribbean nations.
Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.
The outer bands of the hurricane, which was still at Category 3, produced rain, winds and storm 

In [98]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 55, 
     "impactUnit" : "houses damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.', 
                     'Although 55 homes suffered minor damages, the fishing industry was particularly hard hit, with over 200 vessels damaged or destroyed, together with fishing industry infrastructure, disrupting the livelihoods of the coastal communities.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 25,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Agricultural Infrastructure",
     "impactValue" : 200, 
     "impactUnit" : "vessels", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.', 
                     'Although 55 homes suffered minor damages, the fishing industry was particularly hard hit, with over 200 vessels damaged or destroyed, together with fishing industry infrastructure, disrupting the livelihoods of the coastal communities.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 25,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },  
    {"reportDate": "2024-08-06",
     "impactSubtype" : "AOther Economic and Livelihood Impacts",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Barbados was spared a direct hit from the hurricane, as it shifted direction less than 72 hours before landfall.', 
                     'Although 55 homes suffered minor damages, the fishing industry was particularly hard hit, with over 200 vessels damaged or destroyed, together with fishing industry infrastructure, disrupting the livelihoods of the coastal communities.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 25,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1600, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'In Grenada, more than 1,600 people were forced into shelters, with 98% of buildings on Carriacou and Petit Martinique islands suffering severe damage.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },  
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : 98, 
     "impactUnit" : "% buildings damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'In Grenada, more than 1,600 people were forced into shelters, with 98% of buildings on Carriacou and Petit Martinique islands suffering severe damage.'],
     "country" : ["Grenada"],
     "location" : ["Carriacou islands", "Petit Martinique island"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    },  
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Transportation Infrastructure",
     "impactValue" : None, 
     "impactUnit" : "airport", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Power and Energy Production Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'The destruction extended to critical infrastructure, including healthcare facilities and the airport terminal, as well as electrical and water utilities.'],
     "country" : ["Grenada"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 90, 
     "impactUnit" : "% of homes damaged", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services.'],
     "country" : ["Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Access to Healthcare",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services.'],
     "country" : ["Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Service Access Impacts",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['On July 1, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.', 
                     'Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services.'],
     "country" : ["Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Agricultural impact",
     "impactValue" : 1000000000, 
     "impactUnit" : 'USD', 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The agricultural sector alone suffered losses estimated at USD 1 billion, severely affecting food security and local economies.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Agricultural impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.'],
     "country" : ["Jamaica"],
     "location" : ["Clarendon", "St. Elizabeth", "St. Thomas", "Manchester", "Westmoreland", "Hanover"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Jamaica experienced widespread damage as Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.', 
                     'The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.'],
     "country" : ["Jamaica"],
     "location" : ["Clarendon", "St. Elizabeth", "St. Thomas", "Manchester", "Westmoreland", "Hanover"],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The agricultural sector alone suffered losses estimated at USD 1 billion, severely affecting food security and local economies.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of homes and critical infrastructure has led to a significant displacement crisis, with many seeking refuge in temporary shelters.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of homes and critical infrastructure has led to a significant displacement crisis, with many seeking refuge in temporary shelters.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Water, Sanitation, and Hygiene Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Water Quality and Availability",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.', 
                     'The psychological impact on survivors is also profound, with many experiencing trauma and stress, necessitating comprehensive mental health and psychosocial support services.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Epidemic"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Human Health and Wellbeing",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['This displacement, combined with damaged water and sanitation facilities, poses a heightened risk of waterborne diseases, including leptospirosis and cholera.', 
                     'The psychological impact on survivors is also profound, with many experiencing trauma and stress, necessitating comprehensive mental health and psychosocial support services.'],
     "country" : ["Jamaica"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Epidemic"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Recreation, Tourism, and Culture",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic toll of Hurricane Beryl is substantial, with extensive damage to key sectors, such as agriculture, fishing, and tourism.'],
     "country" : ["Jamaica", "Barbados", "Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Agricultural Impacts",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic toll of Hurricane Beryl is substantial, with extensive damage to key sectors, such as agriculture, fishing, and tourism.'],
     "country" : ["Jamaica", "Barbados", "Saint Vincent and the Grenadines"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Other Economic and Livelihood Impacts",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of fishing vessels in Barbados has disrupted the livelihoods of thousands, exacerbating food insecurity and economic instability.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    }, 
    {"reportDate": "2024-08-06",
     "impactSubtype" : "Access to Food",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The destruction of fishing vessels in Barbados has disrupted the livelihoods of thousands, exacerbating food insecurity and economic instability.'],
     "country" : ["Barbados"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm"]
    } 
]

In [99]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRIQ014

In [100]:
iappeal = "MDRIQ014"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRIQ014: 3   2021-12-27
Name: reportDate, dtype: datetime64[ns]
P a g e | 1 Internal DREF Operation n° MDRIQ014 Glide n°: FL-2021-000208-IRQ Date of issue: 27/12/2021 Expected timeframe: 4 months Expected end date: 30/04/2022 Category allocated to the of the disaster or crisis: Yellow DREF allocated: CHF 225,874 Total number of people affected: 7,500+ Number of people to be assisted: 7,500 (1,250 families) Provinces affected: Erbil & Kirkuk Provinces/Regions targeted: Erbil & Kirkuk Operating National Society presence (n° of volunteers, staff, branches): The Iraqi Red Crescent Society (IRCS) is a voluntary humanitarian organization; IRCS has a strong branch network in the country, which is capable of providing relief in times of disasters/emergencies.
Red Cross Red Crescent Movement partners actively involved in the operation: The International Federation of Red Cross and Red Crescent Societies (IFRC) is actively supporting the IRCS in developing the Emergency Plan of Action (EPOA) fo

In [101]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 7500, 
     "impactValueMax" : None, 
     "annotation" : ['P a g e 1 Internal DREF Operation n MDRIQ014 Glide n FL-2021-000208-IRQ Date of issue 27/12/2021 Expected timeframe 4 months Expected end date 30/04/2022 Category allocated to the of the disaster or crisis Yellow DREF allocated CHF 225,874 Total number of people affected 7,500+ Number of people to be assisted 7,500 (1,250 families) Provinces affected Erbil & Kirkuk Provinces/Regions targeted Erbil & Kirkuk Operating National Society presence (n of volunteers, staff, branches) The Iraqi Red Crescent Society (IRCS) is a voluntary humanitarian organization IRCS has a strong branch network in the country, which is capable of providing relief in times of disasters/emergencies.'],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 27,
     "endYear" : 2022,
     "endMonth" : 4,
     "endDay" : 30,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The incident caused widespread damage to houses, infrastructure, and vehicles.', 
                     'The heavy rains overnight caused a flash flood in Erbil, the regions capital, and Kirkuk Governorate, located in Northern Iraq.', 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region.",],
     "country" : ["Iraq"],
     "location" : ["Erbil", "Kirkuk"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood","Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ["Muddy water swept into peoples homes in Erbil's Daratu, Qushtapa, Shamamk, Zhyan, Roshinbiri, and Bahrka neighbourhoods in the early hours of the morning, forcing individuals out of their houses.", 
                     "On 17 December 2021, heavy rainfalls hit the country's northern Kurdish region."],
     "country" : ["Iraq"],
     "location" : ["Erbil's Daratu neighbourhood", "Qushtapa neighbourhood", "Shamamk neighbourhood", "Zhyan neighbourhood", "Roshinbiri neighbourhood", "Bahrka neighbourhood"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 14, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['According to the Kurdish region government on 19 December, 14 people were reported dead by the floods and Emergency Plan of Action (EPoA) Iraq Flash Floods Figure 1 Flooding in Erbil governorate, Iraq (Photo IRCS) P a g e 2 Internal more than 7,000 people are affected by these floods, while IRCS carried further rapid assessments to confirm with the affected families.', 'IRCS carried out further rapid assessments to confirm the number of casualties and affected families , reaching a total of 14 casualties and 7,500 people affected 1,250 families .'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12, 
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    },  
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 7000, 
     "impactValueMax" : 7500, 
     "annotation" : ['According to the Kurdish region government on 19 December, 14 people were reported dead by the floods and Emergency Plan of Action (EPoA) Iraq Flash Floods Figure 1 Flooding in Erbil governorate, Iraq (Photo IRCS) P a g e 2 Internal more than 7,000 people are affected by these floods, while IRCS carried further rapid assessments to confirm with the affected families.', 
                     'IRCS carried out further rapid assessments to confirm the number of casualties and affected families , reaching a total of 14 casualties and 7,500 people affected 1,250 families .'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    },  
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Missing People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['More are feared missing while search and rescue operations are ongoing.'],
     "country" : ["Iraq"],
     "location" : ["Kurdish region"],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 19,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Affected People",
     "impactValue" : 7000000, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The dis astrous heavy rains came at a time when Iraq was already suffering from severe droughts, w ith seven million Iraqis already affected along with the majority of agricultural lands .'],
     "country" : ["Iraq"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Drought"]
    }, 
    {"reportDate": "2021-12-27",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The dis astrous heavy rains came at a time when Iraq was already suffering from severe droughts, w ith seven million Iraqis already affected along with the majority of agricultural lands .'],
     "country" : ["Iraq"],
     "location" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Drought"]
    }
]

In [102]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRSV012

In [103]:
iappeal = "MDRSV012"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print(reports_to_label.loc[reports_to_label["appealCode"]==iappeal]["reportLink"].iloc[0])
print_report(iappeal, ireport)

https://adore.ifrc.org/Download.aspx?FileId=243784
MDRSV012: 5   2019-06-26
Name: reportDate, dtype: datetime64[ns]
DREF Operation n° MDRSV012 GLIDE: n° TC-2018-000167-SLV Date of issue: 25 June 2019 Date of disaster: 15 October 2018 Operation start date: 1 December 2018 Operation end date: 15 February 2019 DREF allocated: 150,671 Swiss francs (CHF) Number of people affected: 7,085 (1,417 families) Number of people assisted: 2,090 (418 families) Host National Society presence (n° of volunteers, staff, branches): The Salvadorean Red Cross Society (SRCS) has one headquarter, 63 branches throughout the country, 2,239 volunteers and 275 staff.
75 volunteers have been trained as National Intervention Teams (NITs) with different specialties (Water, Sanitation and Hygiene Promotion, Logistics, General, ZIKA and Vector Control, Psychosocial Support (PSS)) and 35 active volunteers trained in the Damage Assessment and Needs Analysis (DANA) assessment tool.
Red Cross Red Crescent Movement partner

In [104]:
labelled_impact_reports_dict["MDRSV012"]=[
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 7085, 
     "impactUnit" : "people", 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['DREF Operation n° MDRSV012 GLIDE: n° TC-2018-000167-SLV Date of issue: 25 June 2019 Date of disaster: 15 October 2018 Operation start date: 1 December 2018 Operation end date: 15 February 2019 DREF allocated: 150,671 Swiss francs (CHF) Number of people affected: 7,085 (1,417 families) Number of people assisted: 2,090 (418 families) Host National Society presence (n° of volunteers, staff, branches): The Salvadorean Red Cross Society (SRCS) has one headquarter, 63 branches throughout the country, 2,239 volunteers and 275 staff.'],
     "country" : ["El Salvador"],
     "location" : ["El Salvador"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 15,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : 24, 
     "impactUnit" : 'highways', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : 31, 
     "impactUnit" : 'roads', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 14, 
     "impactUnit" : 'people', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : 'people', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 1090, 
     "impactUnit" : 'people', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Informal settlements",
     "impactValue" : 13, 
     "impactUnit" : 'shelters', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 42, 
     "impactUnit" : 'trees', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Other Transportation Infrastructure",
     "impactValue" : 5, 
     "impactUnit" : 'vehicles', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 6, 
     "impactUnit" : 'Affected homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 1409, 
     "impactUnit" : 'Flooded homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }, 
    {"reportDate": "2019-06-26",
     "impactSubtype" : "Residential Buildings",
     "impactValue" : 2, 
     "impactUnit" : 'Destroyed homes', 
     "impactValuePrecision" : 'exact', 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No.', 
                     'On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazn department and two in La Union department.', 
                     'The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel the cantons of San Felipe and Las Tunas in La Unin department the cantons of Capitn Lazo and Puerto Parada in the municipality of Usulutn as well as the canton of Metalo in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador Floods Photo Area affected by Hurricane Michael.', 
                     'Affected transit routes TOTAL Flooded highways 1 Flooded roads 3 Affected highways 24 Affected rods 31 Isolated communities 1 Affected people TOTAL Injured 14 Dead 4 Sheltered 1,090 Active shelters 13 Other TOTAL Fallen trees 42 Branches of fallen trees 5 Landslides 74 Floods 1 Overflowed rivers 7 Subsidences 1 Vehicles directly affected by the event 5 Affected homes and buildings TOTAL Affected homes 6 Flooded homes 1,409 Destroyed homes 2 Other buildings affected 1 Other buildings destroyed collapsed walls 6 Summary of response Overview of Host National Society The Salvador ean Red Cross Society SRCS constantly monitored the situation through its branches across the country since the onset of lowpressure system No .'],
     "country" : ["El Salvador"],
     "location" : ["Morazn department", "La Union department", "El Brazo canton", "La Canoa canton", "El Tecomatal canton", "San Miguel municipality", "San Felipe canton", "Las Tunas canton", "Capitn Lazo canton", "Puerto Parada canton", "Usulutn municipality", "Metalo canton", "Sonsonate department"],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 5,
     "endYear" : 2019,
     "endMonth" : 10,
     "endDay" : 9,
     "hazards" : ["Tropical storm", "Flood", "Mass Movement"],
    }
]

In [105]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRMY003

In [110]:
iappeal = "MDRMY003"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRMY003: 7   2017-11-21
Name: reportDate, dtype: datetime64[ns]
DREF operation n° MDRMY003 Glide n° FL-2017-000002-MYS Date of issue: 2 November 2017 Date of disaster: 23 January 2017 Operation start date: 8 February 2017 Operation end date: 31 July 2017 N° of people assisted: 15,000 (3,000 families) Amount allocated from DREF: CHF 73,239 Host National Society presence: The Malaysian Red Crescent Society (MRCS) has 160 staff and 230,000 registered volunteers throughout Malaysia’s 13 states and 3 federal territories, including Kelantan, Johor, Pahang and Terengganu which were covered by the DREF operation.
Red Cross Red Crescent Movement partners involved in the operation: The operation was mainly supported by the International Federation Red Cross and Red Crescent Societies (IFRC).
The Singapore Red Cross provided a donation of SGD 20,000, on bilateral basis, and deployed two volunteers for peer-to-peer support.
Other partner organizations actively involved in the operation: The Malay

In [111]:
labelled_impact_reports_dict["MDRMY003"]=[
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 23000, 
     "impactValueMax" : None, 
     "annotation" : ['More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres.', 
                     'Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor and Sabah in East Malaysia.'],
     "country" : ["Malaysia"],
     "location" : ["Johor", "Kelantan", "Pahang", "Perak", "Terengganu", "Malacca", "Selangor", "Sabah"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 8000, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2017-11-21",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 1, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.'],
     "country" : ["Malaysia"],
     "location" : ["Johor"],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }
]

In [112]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRKE058

In [113]:
iappeal = "MDRKE058"
ireport = reports_to_label.loc[reports_to_label["appealCode"]==iappeal]
print_report(iappeal, ireport)

MDRKE058: 1   2024-04-26
Name: reportDate, dtype: datetime64[ns]
1 OPERATION UPDATE Kenya, Africa| Floods Emergency appeal №: MDRKE058 Emergency appeal launched: 23/11/2023.
SITUATION ANALYSIS Communities in Kenya are once again facing heavy rains and devasting floods.
Since mid-March it is reported that at least 38 people have died, 27 people injured and 17 missing as of 23 April.
The above-average rainfall during this March-April-May (MAM) long rains season has severely hit parts of the Lake Victoria Basin, Highlands West of the Rift Valley, Central, Northern and Southern Rift Valley, Highlands East of the Rift Valley (including Nairobi County), Northeastern, Southeastern Lowlands, and Northwestern regions.
The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed.
The Kenya Meteorological Service forecasts that rainfall will peak this week.
The excessive rains already cause havoc in the country where several lives have been lost a

In [114]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 281880, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Page 1 21 DREF Operation Kenya Floods Evacuation of the a ected families in Garissa County Appeal MDRKE058Country KenyaHazard FloodType of DREF Response Crisis Category OrangeEvent Onset SlowDREF Allocation CHF 749,939 Glide Number People A ected 281,880 peoplePeople Targeted 150,000 people Operation Start Date 20231111Operation Timeframe 4 monthsOperation End Date 20240331DREF Published 20231115 Targeted Areas Tana River, Garissa, Wajir, Mandera, Marsabit, Isiolo, MeruPage 2 21 Description of the Event KRCS oods damage What happened, where and when?',],
     "country" : ["Kenya"],
     "location" : ["Tana River", "Garissa", "Wajir","Mandera", "Marsabit", "Isiolo", "Meru"],
     "startYear" : 2023,
     "startMonth" : 11,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 3,
     "endDay" : 31,
     "hazards" : ["Flood"]
    }, 
    
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 25030, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['3 Description of the crisis The country has been experiencing flood affects since the onset of the March-April-May (MAM) long rains season where 24 out of 47 counties have been affected, 25,030 households were affected and 11,275 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 11275, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['3 Description of the crisis The country has been experiencing flood affects since the onset of the March-April-May (MAM) long rains season where 24 out of 47 counties have been affected, 25,030 households were affected and 11,275 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Road Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 7, 
     "impactUnit" : "health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 18, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 80, 
     "impactUnit" : "businesses", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['The economic activities of the areas have been significantly disrupted with major roads destroyed totalling to 11 critical facilities such as 7 health facilities flooded, 18 schools destroyed, 80 businesses hampered.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : None, 
     "impactUnit" : "livestock", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 3401, 
     "impactValueMax" : None, 
     "annotation" : ['More than 3,401 livestock have been lost and approximately 26,748 acres of crops destroyed.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 26748, 
     "impactUnit" : "acres of crops", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['More than 3,401 livestock have been lost and approximately 26,748 acres of crops destroyed.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 31015, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['In Nairobi County, 31,015 people (6,203 households) in the informal settlements of Kware, Kibra, Viwandani, Mukuru Kwa Njenga, Kayole, and Mukuru Kwa Reuben were affected by flooding due to poor and blocked drainage systems.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi County", "Kware", "Kibra", "Viwandani", "Mukuru Kwa Njenga", "Kayole", "Mukuru Kwa Reuben"],
     "startYear" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 178, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 242, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 1000, 
     "impactValueMax" : None, 
     "annotation" : ['5 The floods are exacerbating the humanitarian crisis in the region just as it emerges from the El Nino which occurred late 2023 when at least 178 people were killed, injured 242 and thousands displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : None, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 139051, 
     "impactValueMax" : None, 
     "annotation" : ['Over 139,051 households were affected, 64,516 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "approx", 
     "impactValueMin" : 64516, 
     "impactValueMax" : None, 
     "annotation" : ['Over 139,051 households were affected, 64,516 households displaced.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Infrastructure impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Destroyed infrastructure, health and educational services and facilities.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactValuePrecision" : None, 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Communities reported significant loss of livestock, crops and small businesses leading to loss of livelihood.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 242, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Injured People",
     "impactValue" : 27, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 178, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 38, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 64519, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Displaced People",
     "impactValue" : 11275, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 139071, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected People",
     "impactValue" : 25030, 
     "impactUnit" : "households", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 59, 
     "impactUnit" : "Health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Healthcare Infrastructure",
     "impactValue" : 7, 
     "impactUnit" : "Health facilities", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 29, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Education Infrastructure",
     "impactValue" : 24, 
     "impactUnit" : "schools", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : 17392, 
     "impactUnit" : "livestocks", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Affected Livestock and Animals",
     "impactValue" : 4824, 
     "impactUnit" : "livestocks", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Crop Production and Forestry",
     "impactValue" : 27717, 
     "impactUnit" : "acers of crops", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 1300, 
     "impactUnit" : "business", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : None,
     "endYear" : 2023,
     "endMonth" : 12,
     "endDay" : None,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Other Economic & Livelihood impact",
     "impactValue" : 264, 
     "impactUnit" : "business", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.'],
     "country" : ["Kenya"],
     "location" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 4,
     "endDay" : 23,
     "hazards" : ["Flood", "Mass movement"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 33, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Dadaab"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 22, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Lagdera"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 8, 
     "impactUnit" : "measles cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['October November December short rains season March April May long rains season (reported by 23 April 2024) Counties affected from total 47 38 24 Deaths and injuries 178 (242) 38 (27) Households affected 139,071 25,030 Households displaced 64,519 11,275 Health facilities 59 7 Schools 29 24 Livestock lost 17,392 4,824 Crops lots (acers) 27,717 Small business lost 1,300 264 Camps established 208 49 Disease Outbreak According to the Ministry of Health Kenya, Garissa and Tana River Counties have reported measles outbreak with a total of sixty-three (63) cases reported so far from January.', 
                     'The cases are drawn from Dadaab (33), Lagdera (22), and Garsen (8) sub-counties from Garissa and Tana River respectively.'],
     "country" : ["Kenya"],
     "location" : ["Garsen"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 329, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Lamu countie"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 49, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 29, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Tana River county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    }, 
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 1, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Kiambu county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Infected and Ill People",
     "impactValue" : 1, 
     "impactUnit" : "cholera cases", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['6 The Ministry of Health indicates that 409 cholera cases have been reported in Lamu (329), Nairobi (49), Tana River (29), Kiambu (1), Isiolo (1) counties.'],
     "country" : ["Kenya"],
     "location" : ["Isiolo county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    },  
    {"reportDate": "2024-04-26",
     "impactSubtype" : "Human Deaths",
     "impactValue" : 4, 
     "impactUnit" : "people", 
     "impactValuePrecision" : "exact", 
     "impactValueMin" : None, 
     "impactValueMax" : None, 
     "annotation" : ['Four fatalities have been reported with a case fatality rate of 0.98%.'],
     "country" : ["Kenya"],
     "location" : ["Nairobi county", "Tana River county", "Kiambu county", "Isiolo county"],
     "startYear" : 2024,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazards" : ["Flood", "Epidemic"]
    }, 
]

In [115]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = f"labelled_impact_{iappeal}.csv"
df_impact.to_csv(DATA_LABELLED+fn, index=False)